# Exploratory Data Analysis

---
# Introduction

*Centers for Medicare & Medicaid Services (CMS)* is a U.S. federal government agency responsible for administering major public healthcare programs, including Medicare.

**Medicare** is a U.S. federal health insurance program primarily serving people aged 65 and older. Eligibility can also extend to certain people under 65 with qualifying disabilities, End-Stage Renal Disease (ESRD), or Amyotrophic Lateral Sclerosis (ALS).

Medicare is divided into four parts. This project focuses on **Medicare Part B fee-for-service (FFS) data**.

**Medicare Part B** covers a wide range of medically necessary services provided by physicians and other healthcare professionals, including physician services, outpatient services, preventive services, and diagnostic services. Beneficiaries generally pay a monthly Part B premium and an annual deductible. After the deductible, beneficiaries generally pay 20% of the Medicare-approved amount for covered services when the applicable Medicare payment rules are met.

This project uses the **CMS Medicare Physician & Other Practitioners by Provider and Service** dataset to analyze provider-, service-, specialty-, and regional-level patterns in Medicare healthcare services and payments.


## Key Questions

This project focuses on three main questions at the **specialty, regional, and provider peer-group levels**.

### Question 1 — Specialty Level

**How does the gap between provider-submitted charges and Medicare-allowed amounts vary across different specialties?**

Providers from different specialties may have substantially different submitted charges relative to the amounts allowed by Medicare. This question examines the disparity between provider-submitted charges and Medicare-allowed amounts across specialties.

### Question 2 — Regional Level

**What factors contribute to differences in Medicare expenditures across geographic regions?**

How do Medicare payments and service volumes vary across geographic regions, and to what extent are regional differences associated with payment per service, service volume, or both?

### Question 3 — Provider Peer-Group Level

**Which providers have substantially higher submitted charges than their peers for comparable services performed in the same setting?**

Providers performing the same services in similar settings can still have different billing patterns. This question compares providers within appropriate peer groups to identify unusually high or low submitted-charge patterns and narrow down potential outliers for further investigation.


## Objective

The objective of this project is to analyze CMS Medicare Physician & Other Practitioners data to identify and understand patterns in healthcare charges, Medicare payments, service volumes, and provider billing behavior.

The analysis focuses on three levels: **specialty, geographic region, and provider peer groups**. By examining these levels, the project aims to identify meaningful differences in submitted charges, Medicare expenditures, and provider-level billing patterns, while highlighting potential outliers for further investigation.

The results will be supported by a reproducible data pipeline that transforms and validates the raw CMS data before producing the analytical outputs.


---
# Data Loading

The raw CMS dataset is stored in the project's `data/raw/` directory. The following code locates the project root dynamically so that the notebook can be run from different working directories on both Windows and Linux.

Only the variables required for this project's analysis are loaded from the raw dataset. The complete filtered dataset is retained as `df_full` for source-level profiling, while a reproducible random sample of 200,000 rows is created as `df` for exploratory analysis and visualization.

A fixed random seed is used so that the same sample can be reproduced across runs.


In [1]:
import pandas as pd
from pathlib import Path
import os

SAMPLE_SIZE = 200_000

# Navigate up from wherever Jupyter is to find the project root
PROJECT_ROOT = Path(os.getcwd())
while not (PROJECT_ROOT / 'data').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

FILE_PATH = PROJECT_ROOT / 'data' / 'raw' / 'Medicare_Physician_Other_Practitioners_by_Provider_and_Service_2022.csv'

df_full = pd.read_csv(
    FILE_PATH,
    usecols=[
        'Rndrng_NPI', 'Rndrng_Prvdr_Type', 'Rndrng_Prvdr_State_Abrvtn',
        'Rndrng_Prvdr_Ent_Cd', 'HCPCS_Cd', 'Place_Of_Srvc',
        'Tot_Benes', 'Tot_Srvcs', 'Tot_Bene_Day_Srvcs',
        'Avg_Sbmtd_Chrg', 'Avg_Mdcr_Alowd_Amt', 'Avg_Mdcr_Pymt_Amt', 'Avg_Mdcr_Stdzd_Amt'
    ],
    low_memory=False
)

# Now take a proper random sample
df = df_full.sample(n=SAMPLE_SIZE, random_state=42)

print(f"Full dataset rows: {len(df_full):,}")
print(f"Sample rows: {len(df):,}")
print(f"Columns: {df.shape[1]}")

Full dataset rows: 9,755,427
Sample rows: 200,000
Columns: 13


---
# Dataset Inspection

---
## Initial Dataset Inspection

The full filtered dataset is inspected to establish its basic structure before beginning the exploratory analysis. This includes the number of records, available columns, data types, memory usage, and missing values.

The dataset contains **9,755,427 records across 13 selected columns** and requires approximately **968 MB of memory** in its current pandas representation. No missing values were found in any of the selected columns.

These checks establish the initial data-quality baseline for the project. The observed data types and completeness will also be useful when designing the future ingestion and validation steps of the data pipeline, where schema consistency and unexpected missing values should be explicitly checked.


In [2]:
df.head(3)

,Rndrng_NPI,Rndrng_Prvdr_Ent_Cd,Rndrng_Prvdr_State_Abrvtn,Rndrng_Prvdr_Type,HCPCS_Cd,Place_Of_Srvc,Tot_Benes,Tot_Srvcs,Tot_Bene_Day_Srvcs,Avg_Sbmtd_Chrg,Avg_Mdcr_Alowd_Amt,Avg_Mdcr_Pymt_Amt,Avg_Mdcr_Stdzd_Amt
3351656,1346268067,I,CA,Infectious Disease,99291,F,19,20.0,20,970.000000,233.674500,188.455500,173.877500
4171590,1427314038,I,NV,Diagnostic Radiology,72125,F,13,13.0,13,568.000000,47.860769,29.512308,28.015385
4604675,1477188373,I,TX,Nurse Practitioner,90662,O,17,17.0,17,67.764706,66.465882,66.465882,66.409412


In [3]:
df_full.info()

<class 'pandas.DataFrame'>
RangeIndex: 9755427 entries, 0 to 9755426
Data columns (total 13 columns):
 #   Column                     Dtype  
---  ------                     -----  
 0   Rndrng_NPI                 int64  
 1   Rndrng_Prvdr_Ent_Cd        str    
 2   Rndrng_Prvdr_State_Abrvtn  str    
 3   Rndrng_Prvdr_Type          str    
 4   HCPCS_Cd                   str    
 5   Place_Of_Srvc              str    
 6   Tot_Benes                  int64  
 7   Tot_Srvcs                  float64
 8   Tot_Bene_Day_Srvcs         int64  
 9   Avg_Sbmtd_Chrg             float64
 10  Avg_Mdcr_Alowd_Amt         float64
 11  Avg_Mdcr_Pymt_Amt          float64
 12  Avg_Mdcr_Stdzd_Amt         float64
dtypes: float64(5), int64(3), str(5)
memory usage: 967.6 MB


In [4]:
df_full.isna().sum()

Rndrng_NPI                   0
Rndrng_Prvdr_Ent_Cd          0
Rndrng_Prvdr_State_Abrvtn    0
Rndrng_Prvdr_Type            0
HCPCS_Cd                     0
Place_Of_Srvc                0
Tot_Benes                    0
Tot_Srvcs                    0
Tot_Bene_Day_Srvcs           0
Avg_Sbmtd_Chrg               0
Avg_Mdcr_Alowd_Amt           0
Avg_Mdcr_Pymt_Amt            0
Avg_Mdcr_Stdzd_Amt           0
dtype: int64

---
## Key Profiling Insights & Pipeline Impacts

* **Might be Severe Right Skew (MAD Required):** The distributions exhibit significant upper-tail dispersion and extreme values. This highlights the need for robust statistical methods in provider peer-group analysis, where the median and Median Absolute Deviation (MAD) will be assessed as alternatives to the mean and standard deviation.

* **Database Typing:** Financial metrics might need to be defined as NUMERIC(12, 2) to avoid issues with floating-point rounding. The total services (Tot_Srvcs) should also be defined as NUMERIC(12, 2) because of fractional split billing (minimum value of 2.50). Additionally, the total beneficiaries (Tot_Benes) and total beneficiary days of services (Tot_Bene_Day_Srvcs) should be defined as INTEGER.

* **Privacy Floor Verified:** The minimum observed values for Tot_Benes and Tot_Bene_Day_Srvcs are both 11. This is consistent with the dataset's published minimum-count/suppression rules and should be preserved as an observed source characteristic.

* **Zero Division Risk:** The average Medicare allowed amount (Avg_Mdcr_Alowd_Amt) can hit 0.00. To avoid errors in downstream markup calculations (Avg_Sbmtd_Chrg / Avg_Mdcr_Alowd_Amt), it is necessary to implement a NULLIF guard in SQL to prevent pipeline crashes.

In [5]:
numeric_cols = [
    'Avg_Sbmtd_Chrg', 'Avg_Mdcr_Alowd_Amt', 
    'Avg_Mdcr_Pymt_Amt', 'Avg_Mdcr_Stdzd_Amt',
    'Tot_Benes', 'Tot_Srvcs', 'Tot_Bene_Day_Srvcs'
]
print(df_full[numeric_cols].describe().round(2))

       Avg_Sbmtd_Chrg  Avg_Mdcr_Alowd_Amt  Avg_Mdcr_Pymt_Amt  \
count      9755427.00          9755427.00         9755427.00   
mean           398.99              104.98              81.95   
std           1358.12              312.13             249.16   
min              0.00                0.00               0.00   
25%             69.00               27.32              21.31   
50%            166.67               68.54              52.28   
75%            343.00              120.67              89.22   
max          99999.99            59035.73           47043.99   

       Avg_Mdcr_Stdzd_Amt   Tot_Benes   Tot_Srvcs  Tot_Bene_Day_Srvcs  
count          9755427.00  9755427.00  9755427.00          9755427.00  
mean                80.93       84.27      258.22              134.31  
std                248.43     1301.98     6269.63             2116.34  
min                  0.00       11.00        2.50               11.00  
25%                 19.52       17.00       20.00              

---
### Boundary and Zero-Value Validation Key Points

* **Data Integrity Confirmed (No Negative Values):** All 9.75 million records contain non-negative values for all financial and volume measures. We can confidently apply non-negative check constraints (CHECK (col >= 0)) in the database DDL and dbt schema tests.

* **Proof of Zero-Division Safeguards:** TheAvg_Mdcr_Alowd_Amt field includes exactly 15 rows with a value of zero. When calculating the provider markup ratio (`Avg_Sbmtd_Chrg / Avg_Mdcr_Alowd_Amt`), it is essential to use an explicit `NULLIF(Avg_Mdcr_Alowd_Amt, 0)` safeguard in SQL to prevent pipeline crashes.

`Note`: Avg_Mdcr_Pymt_Amt contains 109 zero-valued records. These observations require awareness when interpreting payment-based metrics and should be investigated against the CMS field definition before being treated as anomalies.

In [6]:
for col in numeric_cols:
    neg_count = (df_full[col] < 0).sum()
    zero_count = (df_full[col] == 0).sum()
    print(f"{col:20s}: {neg_count} negatives | {zero_count} zeroes")

Avg_Sbmtd_Chrg      : 0 negatives | 0 zeroes
Avg_Mdcr_Alowd_Amt  : 0 negatives | 15 zeroes
Avg_Mdcr_Pymt_Amt   : 0 negatives | 109 zeroes
Avg_Mdcr_Stdzd_Amt  : 0 negatives | 25 zeroes
Tot_Benes           : 0 negatives | 0 zeroes
Tot_Srvcs           : 0 negatives | 0 zeroes
Tot_Bene_Day_Srvcs  : 0 negatives | 0 zeroes


---
## Categorical Domain Profiling Takeaways

* **Storage Optimization (Place of Service & Provider Entity Code):** Both attributes have only 2 unique single-character values: 'O' and 'F' for the place of service, and 'I' and 'O' for the entity type. Storing these attributes as CHAR(1) in PostgreSQL optimizes disk storage and improves query performance.
  
* **dbt Schema Test Specification:** The empirical domains directly inform our dbt data contract tests:
  - `Place_Of_Srvc`: `accepted_values: ['O', 'F']`
  - `Rndrng_Prvdr_Ent_Cd`: `accepted_values: ['I', 'O']`
  
* There are 103 unique medical specialties.

In [7]:
cat_cols = ['Rndrng_Prvdr_Type', 'Place_Of_Srvc', 'Rndrng_Prvdr_Ent_Cd']
for col in cat_cols:
    print(f"{col}: {df_full[col].nunique()} unique values")
    print(df_full[col].value_counts().head(5))
    print()

Rndrng_Prvdr_Type: 103 unique values
Rndrng_Prvdr_Type
Diagnostic Radiology    1137764
Internal Medicine        868292
Nurse Practitioner       851917
Family Practice          813473
Physician Assistant      518182
Name: count, dtype: int64

Place_Of_Srvc: 2 unique values
Place_Of_Srvc
O    6233245
F    3522182
Name: count, dtype: int64

Rndrng_Prvdr_Ent_Cd: 2 unique values
Rndrng_Prvdr_Ent_Cd
I    9210142
O     545285
Name: count, dtype: int64



---
## Grain Validation

* **Confirmed Natural Composite Primary Key:** When grouping by `['Rndrng_NPI', 'HCPCS_Cd', 'Place_Of_Srvc']`, there are precisely 9,755,427 unique combinations. This corresponds to all 9,755,427 rows, with no duplicates detected.
* **Database Constraint Specification:** In the PostgreSQL warehouse schema and the downstream dbt models, (rndrng_npi, hcpcs_cd, place_of_srvc) can be designated as the composite primary key. This designation poses no risk of key violation or row fan-out.

In [8]:
grain_cols = ['Rndrng_NPI', 'HCPCS_Cd', 'Place_Of_Srvc']

grain_check = df_full.groupby(grain_cols).size().reset_index(name='row_count')
duplicates = grain_check[grain_check['row_count'] > 1]

print(f"Total rows in dataset:          {len(df_full):,}")
print(f"Unique grain combinations:       {len(grain_check):,}")
print(f"Combinations appearing > 1 time: {len(duplicates):,}")

if len(duplicates) == 0:
    print("\n✓ Grain confirmed — NPI + HCPCS + Place of Service uniquely identifies every row.")
    print("  One row = one provider + one procedure + one setting for the reporting year.")
else:
    print("\n✗ Duplicates found — investigate before schema design.")
    print(duplicates.head(10))

Total rows in dataset:          9,755,427
Unique grain combinations:       9,755,427
Combinations appearing > 1 time: 0

✓ Grain confirmed — NPI + HCPCS + Place of Service uniquely identifies every row.
  One row = one provider + one procedure + one setting for the reporting year.


---
## NPI Validation
### Key Takeaways for National Provider Identifier (NPI) Validation

* **Consistent Federal Format:** All 9,755,427 provider records consist of exactly 10 digits, with no malformed or truncated identifiers.
* **Schema Definition (VARCHAR(10)):** The NPI is a nominal identifier rather than a quantitative measurement. In PostgreSQL DDL and dbt models, it can be stored as `VARCHAR(10)` instead of an integer type. This approach prevents arithmetic evaluation and avoids the accidental removal of leading zeros in future pipeline runs.
* **Reliable Joining:** Keeping the NPI as a uniform 10-character string ensures stable and uncorrupted primary-to-foreign key joins between staging tables, provider dimension tables, and analytical marts.

In [9]:
# NPI should be a 10-digit identifier
# It reads as int64 in pandas — convert to string for length validation
npi_str = df_full['Rndrng_NPI'].astype(str)
npi_lengths = npi_str.str.len().value_counts().sort_index()

print("NPI length distribution (all should be 10):")
print(npi_lengths)
print()

non_10 = (npi_str.str.len() != 10).sum()
print(f"NPIs not exactly 10 digits: {non_10:,}")

if non_10 == 0:
    print("✓ All NPIs are exactly 10 digits.")
else:
    print("✗ Some NPIs are not 10 digits — investigate.")
    print(df_full[npi_str.str.len() != 10][['Rndrng_NPI']].head(10))

print()
print("Note: NPI is an identifier, not a measurement.")
print("It will be stored as VARCHAR(10) or TEXT in PostgreSQL, not as a numeric type.")
print("Arithmetic on NPIs is meaningless — only equality comparisons matter.")

NPI length distribution (all should be 10):
Rndrng_NPI
10    9755427
Name: count, dtype: int64

NPIs not exactly 10 digits: 0
✓ All NPIs are exactly 10 digits.

Note: NPI is an identifier, not a measurement.
It will be stored as VARCHAR(10) or TEXT in PostgreSQL, not as a numeric type.
Arithmetic on NPIs is meaningless — only equality comparisons matter.


---
## HCPCS Procedure Code Validation

* **Fixed-Width Format Confirmed (CHAR(5))**: All 9,755,427 records contain exactly 5-character procedure codes, representing 6,326 unique HCPCS codes. These can be stored as fixed-width CHAR(5) in PostgreSQL.
  
* **Alphanumeric Structure Confirmed**: The presence of Level II codes (e.g., G0008) indicates that procedure codes must remain as strings/text during the ingestion and transformation processes to avoid null values.

In [14]:
# HCPCS codes should be 5 characters — either 5 numeric digits (CPT) 
# or 1 letter + 4 digits (Level II)
hcpcs_lengths = df_full['HCPCS_Cd'].astype(str).str.len().value_counts().sort_index()

print("HCPCS code length distribution (all should be 5):")
print(hcpcs_lengths)
print()

total_unique_hcpcs = df_full['HCPCS_Cd'].nunique()
print(f"Total unique HCPCS codes: {total_unique_hcpcs:,}")
print()

# Check how many HCPCS codes contain letters (non-numeric characters)
alpha_count = (~df_full['HCPCS_Cd'].astype(str).str.isnumeric()).sum()
print(f"HCPCS codes containing letters: {alpha_count:,}")

# Check for non-5-character codes
non_5 = (df_full['HCPCS_Cd'].astype(str).str.len() != 5).sum()
print(f"Codes not exactly 5 characters: {non_5:,}")

if non_5 == 0:
    print("✓ All HCPCS codes are exactly 5 characters.")
else:
    print("✗ Some codes are not 5 characters — investigate.")

HCPCS code length distribution (all should be 5):
HCPCS_Cd
5    9755427
Name: count, dtype: int64

Total unique HCPCS codes: 6,326

HCPCS codes containing letters: 1,053,862
Codes not exactly 5 characters: 0
✓ All HCPCS codes are exactly 5 characters.


---
## Distribution Shape & Skewness Analysis Takeaways

* **Severe Right Skewness Confirmed:** All financial metrics exhibit extreme positive skewness, with values ranging from 27.34 for submitted charges to over 42.9 for Medicare payments. The mean submitted charge is nearly twice the median or more.
* **Significant Disparity in the Top 1% Tail:** While 99% of provider charges are at or below `$4,000.00`, maximum values reach as high as $99,999.99, which significantly skews standard parametric measures.
* **Methodological Justification for Q3:** Standard Gaussian outlier detection methods, such as Z-scores and standard deviation, will fail due to the variance inflation caused by outliers. Therefore, the Median Absolute Deviation (MAD) is necessary for modeling anomalies within peer groups in dbt.

In [13]:
print("Distribution shape analysis — full dataset:")
print(f"{'Column':<25} {'Mean':>10} {'Median':>10} {'Skewness':>10} {'P99':>12}")
print("-" * 70)

for col in ['Avg_Sbmtd_Chrg', 'Avg_Mdcr_Alowd_Amt', 
            'Avg_Mdcr_Pymt_Amt', 'Avg_Mdcr_Stdzd_Amt']:
    mean = df_full[col].mean()
    median = df_full[col].median()
    skew = df_full[col].skew()
    p99 = df_full[col].quantile(0.99)
    print(f"{col:<25} {mean:>10.2f} {median:>10.2f} {skew:>10.2f} {p99:>12.2f}")

print()
print("Interpretation: skewness > 1 indicates right skew.")
print("Skewness > 2 or > 3 means extreme right skew where mean >> median.")
print("This directly informs whether robust statistics (MAD) are appropriate for Q3.")

Distribution shape analysis — full dataset:
Column                          Mean     Median   Skewness          P99
----------------------------------------------------------------------
Avg_Sbmtd_Chrg                398.99     166.67      27.34      4000.00
Avg_Mdcr_Alowd_Amt            104.98      68.54      42.96       816.61
Avg_Mdcr_Pymt_Amt              81.95      52.28      42.95       650.40
Avg_Mdcr_Stdzd_Amt             80.93      52.91      43.49       636.60

Interpretation: skewness > 1 indicates right skew.
Skewness > 2 or > 3 means extreme right skew where mean >> median.
This directly informs whether robust statistics (MAD) are appropriate for Q3.
